# InterLeaf AI Pipeline — Predictions Only

This notebook performs **prediction only**.

It reads `Reddit_250_Prediction_Input.csv`, which contains only:
- `post_id`
- `post_text`

It does **not** read the gold annotation workbook.

Pipeline:

1. Narrative Normalization
2. Harm and Safety Screening
3. LLM-Based Contextual Tagging
4. Ontology-Based Keyword Tagging
5. Embedding Similarity Matching
6. Confidence Fusion and Final Tag Selection

Output: `reddit_250_predictions.jsonl`

Keep this notebook frozen once the evaluation run begins.

In [1]:
# Colab install cell, if needed:
# !pip install -q -U google-genai pandas numpy tqdm

In [16]:
import os
import re
import json
import time
import getpass
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from google import genai
from google.genai import types


INPUT_CSV = Path("Reddit_250_Prediction_Input.csv")
OUTPUT_JSONL = Path("reddit_250_predictions.jsonl")

# Keep the same models used by the original pipeline unless you intentionally
# preregister a different version before running the experiment.
LLM_MODEL = "gemini-2.5-flash"
EMBED_MODEL = "gemini-embedding-001"

SAFETY_THRESHOLD = 0.80
SIMILARITY_THRESHOLD = 0.60
FINAL_TAG_THRESHOLD = 0.60

MAX_CONTEXTUAL_CANDIDATES = 15
MAX_FINAL_TAGS = 5
AGREEMENT_BONUS = 0.10

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Gemini API key: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

print("Gemini initialized")
print("LLM:", LLM_MODEL)
print("Embedding:", EMBED_MODEL)

Gemini initialized
LLM: gemini-2.5-flash
Embedding: gemini-embedding-001


In [3]:
df = pd.read_csv(INPUT_CSV)

required = {"post_id", "post_text"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

if len(df) != 250:
    print(f"Warning: expected 250 posts, found {len(df)}")

if df["post_id"].duplicated().any():
    raise ValueError("Duplicate post_id values found.")

print(f"Loaded {len(df)} prediction inputs")
print(df.head(2))

Loaded 250 prediction inputs
  post_id                                          post_text
0    R153  Let me start by saying that I'm not officially...
1    R180  Peripheral neuropathy sufferers on gabapentin....


## Ontology

In [4]:
ONTOLOGY = {
    "body_area": [
        "head", "neck", "shoulder", "back", "hip",
        "knee", "hands", "feet", "diffuse"
    ],
    "symptom": [
        "burning", "stabbing", "throbbing", "stiffness",
        "fatigue", "brain fog", "sleep trouble", "numbness", "tingling"
    ],
    "trigger": [
        "weather", "stress", "long sitting", "hormonal cycle",
        "certain foods", "overexertion", "poor sleep"
    ],
    "coping": [
        "pacing", "heat", "mindfulness", "medication",
        "physical therapy", "stretching", "TENS", "CBT skills", "rest breaks"
    ],
    "emotion": [
        "grief", "anger", "frustration", "shame",
        "hope", "acceptance", "overwhelm", "loneliness"
    ],
}

ONTOLOGY_CATEGORIES = set(ONTOLOGY)

def canonical(x):
    return re.sub(r"\s+", " ", str(x).strip().lower())

def ontology_items():
    return [
        {
            "category": category,
            "label": label,
            "key": f"{category}:{canonical(label)}",
        }
        for category, labels in ONTOLOGY.items()
        for label in labels
    ]

ONTOLOGY_ITEMS = ontology_items()

## 1. Narrative Normalization

In [5]:
def normalize_narrative(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return ""

    text = str(raw).strip()

    # Privacy-oriented masking and whitespace cleanup only.
    # Do not rewrite spelling, grammar, or narrative style.
    text = re.sub(r"https?://\S+|www\.\S+", "[URL]", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", "[EMAIL]", text)
    text = re.sub(r"(?<!\w)[uU]/[A-Za-z0-9_-]+", "[USER]", text)
    text = re.sub(r"(?<!\w)@[A-Za-z0-9_]+", "[USER]", text)
    text = re.sub(r"(?<!\w)[rR]/[A-Za-z0-9_]+", "[SUBREDDIT]", text)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

## Shared Gemini JSON helper

In [6]:
def generate_json(prompt, attempts=3):
    last_error = None

    for attempt in range(attempts):
        try:
            response = client.models.generate_content(
                model=LLM_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                ),
            )
            return json.loads(response.text)

        except Exception as e:
            last_error = e
            if attempt < attempts - 1:
                time.sleep(2 ** attempt)

    raise last_error

## 2. Harm and Safety Screening

In [7]:
def safety_screen(text):
    if not text:
        return {
            "label": "not_harmful",
            "score": 0.0,
            "categories": [],
            "evidence": "",
        }

    prompt = f'''
Classify safety-critical content in this chronic-pain narrative.

HARMFUL:
- current or recent suicidal/self-harm thoughts, wishes, plans, attempts, or behavior;
- conditional suicide or self-harm plans;
- credible violence toward another person;
- active safety-relevant abuse or coercion;
- targeted hate or advocacy of harm;
- dangerous medication misuse, overdose, or acute self-endangering medical behavior.

Passive death wishes and recent suicide attempts count as harmful.
The concern may involve the writer or another person described in the post.

NOT HARMFUL:
severe pain, depression, frustration, ordinary medication questions, or phrases
such as "I can't go on" when the text does not actually imply death or self-harm.

Return JSON only:
{{
  "safety_score": 0.0,
  "categories": [],
  "evidence": ""
}}

Categories may only be:
["self_harm", "violence", "abuse", "hate", "medical_risk"].

Evidence must be a short exact quote, or "" if none.

NARRATIVE:
{text}
'''.strip()

    try:
        result = generate_json(prompt)

        score = float(result.get("safety_score", 0.0))
        score = max(0.0, min(1.0, score))

        allowed = {"self_harm", "violence", "abuse", "hate", "medical_risk"}
        categories = [x for x in result.get("categories", []) if x in allowed]

        return {
            "label": "harmful" if score >= SAFETY_THRESHOLD else "not_harmful",
            "score": score,
            "categories": categories,
            "evidence": str(result.get("evidence", "")).strip(),
        }

    except Exception as e:
        return {
            "label": None,
            "score": None,
            "categories": [],
            "evidence": "",
            "error": str(e),
        }

## 3. LLM-Based Contextual Tagging

In [8]:
def contextual_tagging(text):
    if not text:
        return []

    prompt = f'''
Extract up to {MAX_CONTEXTUAL_CANDIDATES} salient tags from this chronic-pain narrative.

For each tag return:
- label: short phrase
- category: body_area, symptom, trigger, coping, emotion, or other
- confidence: 0 to 1
- evidence: short exact quote

Rules:
- tag only what the narrative supports;
- do not force a predefined ontology label;
- use "other" for important context outside the ontology;
- avoid duplicate or near-duplicate tags.

Return JSON only:
{{
  "candidate_tags": [
    {{
      "label": "",
      "category": "",
      "confidence": 0.0,
      "evidence": ""
    }}
  ]
}}

NARRATIVE:
{text}
'''.strip()

    try:
        result = generate_json(prompt)
        candidates = []

        for item in result.get("candidate_tags", [])[:MAX_CONTEXTUAL_CANDIDATES]:
            label = canonical(item.get("label", ""))
            if not label:
                continue

            category = canonical(item.get("category", "other"))
            if category not in ONTOLOGY_CATEGORIES | {"other"}:
                category = "other"

            confidence = float(item.get("confidence", 0.5))
            confidence = max(0.0, min(1.0, confidence))

            candidates.append({
                "label": label,
                "category": category,
                "llm_confidence": confidence,
                "evidence": str(item.get("evidence", "")).strip(),
            })

        return candidates

    except Exception as e:
        print("Contextual tagging error:", e)
        return []

## 4. Ontology-Based Keyword Tagging

In [9]:
KEYWORD_PATTERNS = {
    "body_area": {
        "head": r"\b(head|headache|headaches|migraine|migraines)\b",
        "neck": r"\b(neck|cervical)\b",
        "shoulder": r"\bshoulders?\b",
        "back": r"\b(back|lumbar|lower back|upper back)\b",
        "hip": r"\bhips?\b",
        "knee": r"\bknees?\b",
        "hands": r"\b(hand|hands|finger|fingers|wrist|wrists)\b",
        "feet": r"\b(foot|feet|toe|toes|ankle|ankles)\b",
        "diffuse": r"\b(diffuse|widespread|whole body|full body|all over)\b",
    },
    "symptom": {
        "burning": r"\bburn(ing|s)?\b",
        "stabbing": r"\bstab(bing|s)?\b",
        "throbbing": r"\bthrob(bing|s)?\b",
        "stiffness": r"\bstiff(ness)?\b",
        "fatigue": r"\b(fatigue|fatigued|exhausted|exhaustion)\b",
        "brain fog": r"\bbrain fog\b",
        "sleep trouble": r"\b(insomnia|sleep trouble|trouble sleeping|can't sleep|cannot sleep)\b",
        "numbness": r"\bnumb(ness)?\b",
        "tingling": r"\b(tingling|pins and needles)\b",
    },
    "trigger": {
        "weather": r"\b(weather|barometric|humidity|humid|rain|cold weather|hot weather)\b",
        "stress": r"\bstress(ed|ful)?\b",
        "long sitting": r"\b(long sitting|sitting too long|prolonged sitting)\b",
        "hormonal cycle": r"\b(period|menstrual|menstruation|ovulation|hormonal cycle)\b",
        "certain foods": r"\b(food|foods|dairy|gluten)\b",
        "overexertion": r"\b(overexertion|overdid|overdo|pushed myself|too much activity)\b",
        "poor sleep": r"\b(poor sleep|bad sleep|lack of sleep|sleep deprived|sleep deprivation)\b",
    },
    "coping": {
        "pacing": r"\b(pacing|pace myself|pace my activities)\b",
        "heat": r"\b(heat|heating pad|hot bath|hot shower|warm compress)\b",
        "mindfulness": r"\b(mindfulness|meditation|meditate|breathing exercise)\b",
        "medication": r"\b(medication|medications|medicine|meds|painkiller|painkillers)\b",
        "physical therapy": r"\b(physical therapy|physiotherapy|physio)\b",
        "stretching": r"\b(stretching|stretch)\b",
        "TENS": r"\bTENS\b",
        "CBT skills": r"\b(CBT|cognitive behavioral|cognitive behavioural)\b",
        "rest breaks": r"\b(rest break|rest breaks|take a break|taking breaks|lie down|lying down)\b",
    },
    "emotion": {
        "grief": r"\b(grief|grieve|grieving)\b",
        "anger": r"\b(anger|angry|furious|rage)\b",
        "frustration": r"\b(frustration|frustrated|fed up)\b",
        "shame": r"\b(shame|ashamed)\b",
        "hope": r"\b(hope|hopeful)\b",
        "acceptance": r"\b(acceptance|accepting|accepted)\b",
        "overwhelm": r"\b(overwhelm|overwhelmed)\b",
        "loneliness": r"\b(loneliness|lonely|isolated)\b",
    },
}

NEGATION = re.compile(
    r"\b(no|not|never|without|don't|doesn't|didn't|isn't|wasn't|aren't|weren't)\b",
    re.I,
)

TRIGGER_CUE = re.compile(
    r"\b(trigger|worsen|worse|flare|sets? off|brings? on|aggravat|caus|increase)\w*\b",
    re.I,
)

COPING_CUE = re.compile(
    r"\b("
    r"i use|i used|i take|i took|i try|i tried|i do|i did|"
    r"using|taking|currently on|been on|started|prescribed|"
    r"helps me|helped me|gives me relief"
    r")\b",
    re.I,
)

def sentence_chunks(text):
    return [
        s.strip()
        for s in re.split(r"(?<=[.!?])\s+|\n+", text)
        if s.strip()
    ]

def nearby_negation(sentence, match_start):
    window = sentence[max(0, match_start - 45):match_start]
    return bool(NEGATION.search(window))

def ontology_keyword_tagging(text):
    found = {}

    for sentence in sentence_chunks(text):
        for category, label_patterns in KEYWORD_PATTERNS.items():
            for label, pattern in label_patterns.items():
                match = re.search(pattern, sentence, flags=re.I)

                if not match or nearby_negation(sentence, match.start()):
                    continue

                if category == "trigger" and not TRIGGER_CUE.search(sentence):
                    continue

                if category == "coping" and not COPING_CUE.search(sentence):
                    continue

                key = f"{category}:{canonical(label)}"

                if key not in found:
                    found[key] = {
                        "category": category,
                        "label": label,
                        "key": key,
                        "evidence": sentence,
                    }

    return list(found.values())

## 5. Embedding Similarity Matching

In [10]:
_embedding_cache = {}
ONTOLOGY_VECTORS = None

def embed_text(text):
    key = canonical(text)
    if not key:
        return []

    if key not in _embedding_cache:
        result = client.models.embed_content(
            model=EMBED_MODEL,
            contents=key,
        )
        _embedding_cache[key] = result.embeddings[0].values

    return _embedding_cache[key]

def cosine(a, b):
    if not a or not b or len(a) != len(b):
        return 0.0

    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

def prepare_ontology_embeddings():
    global ONTOLOGY_VECTORS

    if ONTOLOGY_VECTORS is None:
        ONTOLOGY_VECTORS = [
            {**item, "vector": embed_text(item["label"])}
            for item in ONTOLOGY_ITEMS
        ]

    return ONTOLOGY_VECTORS

def embedding_similarity_matching(contextual_candidates):
    ontology_vectors = prepare_ontology_embeddings()
    matched = []

    for candidate in contextual_candidates:
        vector = embed_text(candidate["label"])

        # Preserve the LLM's broad domain when it supplied one.
        if candidate["category"] in ONTOLOGY_CATEGORIES:
            search_space = [
                x for x in ontology_vectors
                if x["category"] == candidate["category"]
            ]
        else:
            search_space = ontology_vectors

        best = None

        for item in search_space:
            sim = cosine(vector, item["vector"])

            if best is None or sim > best["similarity"]:
                best = {
                    "category": item["category"],
                    "label": item["label"],
                    "key": item["key"],
                    "similarity": sim,
                }

        mapped = bool(best and best["similarity"] >= SIMILARITY_THRESHOLD)

        matched.append({
            **candidate,
            "mapped_to_ontology": mapped,
            "canonical_category": best["category"] if mapped else "other",
            "canonical_label": best["label"] if mapped else candidate["label"],
            "canonical_key": best["key"] if mapped else None,
            "embedding_similarity": float(best["similarity"]) if best else 0.0,
        })

    return matched

## 6. Confidence Fusion and Final Tag Selection

In [11]:
def confidence_fusion_and_select(matched_contextual, keyword_tags):
    fused = {}

    for item in matched_contextual:
        if not item["mapped_to_ontology"]:
            continue

        key = item["canonical_key"]
        contextual_score = (
            item["llm_confidence"]
            + item["embedding_similarity"]
        ) / 2.0

        old = fused.get(key)

        if old is None or contextual_score > old["contextual_score"]:
            fused[key] = {
                "key": key,
                "category": item["canonical_category"],
                "label": item["canonical_label"],
                "contextual_score": float(contextual_score),
                "keyword_support": False,
                "evidence": item.get("evidence", ""),
            }

    for item in keyword_tags:
        key = item["key"]

        if key not in fused:
            fused[key] = {
                "key": key,
                "category": item["category"],
                "label": item["label"],
                "contextual_score": 0.0,
                "keyword_support": True,
                "evidence": item.get("evidence", ""),
            }
        else:
            fused[key]["keyword_support"] = True

    ontology_candidates = []

    for item in fused.values():
        score = (
            item["contextual_score"]
            if item["contextual_score"] > 0
            else FINAL_TAG_THRESHOLD
        )

        if item["keyword_support"] and item["contextual_score"] > 0:
            score = min(1.0, score + AGREEMENT_BONUS)

        ontology_candidates.append({
            "tag": item["key"],
            "category": item["category"],
            "label": item["label"],
            "score": float(score),
            "is_ontology": True,
            "evidence": item["evidence"],
            "supported_by_contextual": item["contextual_score"] > 0,
            "supported_by_keyword": item["keyword_support"],
        })

    freeform = {}

    for item in matched_contextual:
        if item["mapped_to_ontology"]:
            continue

        key = canonical(item["label"])
        candidate = {
            "tag": key,
            "category": "other",
            "label": key,
            "score": float(item["llm_confidence"]),
            "is_ontology": False,
            "evidence": item.get("evidence", ""),
        }

        if key not in freeform or candidate["score"] > freeform[key]["score"]:
            freeform[key] = candidate

    freeform_candidates = list(freeform.values())

    all_candidates = ontology_candidates + freeform_candidates
    all_candidates.sort(key=lambda x: x["score"], reverse=True)

    final_tags = [
        x for x in all_candidates
        if x["score"] >= FINAL_TAG_THRESHOLD
    ][:MAX_FINAL_TAGS]

    return {
        "fused_ontology_candidates": sorted(
            ontology_candidates,
            key=lambda x: x["score"],
            reverse=True,
        ),
        "candidate_freeform_tags": sorted(
            freeform_candidates,
            key=lambda x: x["score"],
            reverse=True,
        ),
        "final_tags": final_tags,
        "predicted_ontology_tags": [
            x["tag"] for x in final_tags if x["is_ontology"]
        ],
        "predicted_tag_confidence": {
            x["tag"]: x["score"]
            for x in final_tags
            if x["is_ontology"]
        },
        "final_freeform_tags": [
            {
                "tag": x["tag"],
                "score": x["score"],
                "evidence": x["evidence"],
            }
            for x in final_tags
            if not x["is_ontology"]
        ],
    }

## End-to-End Pipeline

In [12]:
def process_post(post_id, raw_text):
    normalized = normalize_narrative(raw_text)

    safety = safety_screen(normalized)
    contextual = contextual_tagging(normalized)
    keyword = ontology_keyword_tagging(normalized)
    matched = embedding_similarity_matching(contextual)
    fused = confidence_fusion_and_select(matched, keyword)

    return {
        "post_id": str(post_id),
        "safety": safety,
        "contextual_candidates": contextual,
        "keyword_ontology_tags": keyword,
        "matched_contextual_candidates": matched,
        **fused,
    }

## Run all 250 posts

In [17]:
def completed_ids(path):
    if not path.exists():
        return set()

    done = set()

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    done.add(str(json.loads(line)["post_id"]))
                except Exception:
                    pass

    return done


def run_predictions():
    done = completed_ids(OUTPUT_JSONL)

    work = df.copy()
    work["post_id"] = work["post_id"].astype(str)
    remaining = work[~work["post_id"].isin(done)]

    print("Already completed:", len(done))
    print("Remaining:", len(remaining))

    mode = "a" if OUTPUT_JSONL.exists() else "w"

    with OUTPUT_JSONL.open(mode, encoding="utf-8") as out:
        for _, row in tqdm(
            remaining.iterrows(),
            total=len(remaining),
            desc="InterLeaf predictions",
        ):
            record = process_post(
                post_id=row["post_id"],
                raw_text=row["post_text"],
            )

            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()

    print("Saved:", OUTPUT_JSONL)


# Uncomment only when you are ready to start the frozen evaluation run:
run_predictions()

Already completed: 0
Remaining: 250


InterLeaf predictions:   0%|          | 0/250 [00:00<?, ?it/s]

Saved: reddit_250_predictions.jsonl


## Output contract

The evaluation notebook expects these fields in each JSONL record:

- `post_id`
- `safety.score`
- `safety.label`
- `predicted_ontology_tags`
- `predicted_tag_confidence`
- `candidate_freeform_tags`
- `final_freeform_tags`

It may also use intermediate fields for error analysis, but it does not modify predictions.